In [1]:
import time
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [2]:
MAX_PAGES = 5
DELAY_SECONDS = 2.0
TIMEOUT = 20

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; BCIT-CourseMap/1.0; educational)"
}

COURSE_URLS = [
    "https://www.bcit.ca/courses/applied-data-management-for-analytics-babi-4000/",
    "https://www.bcit.ca/courses/computers-and-the-law-blaw-3600/",
    "https://www.bcit.ca/courses/dqm-with-python-babi-4005/",
]

In [3]:
def fetch_html(url):
    r = requests.get(url, timeout=TIMEOUT)  # requests follows redirects automatically
    if r.status_code != 200:
        print("Failed:", r.status_code, url)
        return None, None
    return r.text, r.url  # r.url is the final redirect target

In [4]:
def extract_after_label(page_text, label):
    """
    Finds the first occurrence of 'label' and returns the text after it,
    until the next line that looks like a new section header.
    """
    lines = [line.strip() for line in page_text.split("\n") if line.strip()]

    # find where the label appears
    start_index = None
    for i, line in enumerate(lines):
        if line.lower() == label.lower():
            start_index = i
            break

    if start_index is None:
        return None

    # collect lines after label until a "new section" begins
    collected = []
    for j in range(start_index + 1, len(lines)):
        line = lines[j]

        # stop if we hit another section header we care about
        if line.lower() in [
            "course overview",
            "prerequisite(s)",
            "credits",
            "learning outcomes",
            "related programs",
        ]:
            break

        collected.append(line)

    result = " ".join(collected).strip()
    return result if result else None

In [5]:
def parse_course(html, final_url):
    soup = BeautifulSoup(html, "html.parser")

    # Title
    title_tag = soup.find("h1")
    title = title_tag.get_text(strip=True) if title_tag else None

    # Convert whole page to readable text (line by line)
    page_text = soup.get_text("\n", strip=True)

    # Pull out sections by label (simple + readable)
    overview = extract_after_label(page_text, "Course Overview")
    prereq = extract_after_label(page_text, "Prerequisite(s)")
    credits = extract_after_label(page_text, "Credits")

    # Learning outcomes: grab text after the label (works even if not perfect)
    outcomes_text = extract_after_label(page_text, "Learning Outcomes")

    return {
        "url": final_url,
        "title": title,
        "course_overview": overview,
        "prerequisites": prereq,
        "credits": credits,
        "learning_outcomes": outcomes_text,
    } 

In [6]:
rows = []

for i, url in enumerate(COURSE_URLS, start=1):
    print(f"Fetching {i}/{len(COURSE_URLS)}:", url)

    html, final_url = fetch_html(url)
    if html:
        rows.append(parse_course(html, final_url))

    if i < len(COURSE_URLS):
        time.sleep(DELAY_SECONDS)

df = pd.DataFrame(rows)
df

Fetching 1/3: https://www.bcit.ca/courses/applied-data-management-for-analytics-babi-4000/
Fetching 2/3: https://www.bcit.ca/courses/computers-and-the-law-blaw-3600/
Fetching 3/3: https://www.bcit.ca/courses/dqm-with-python-babi-4005/


,url,title,course_overview,prerequisites,credits,learning_outcomes
0,https://www.bcit.ca/courses/applied-data-manag...,Applied Data Management for AnalyticsBABI 4000,This course will introduce students to the fun...,50% in BABI 3000,5.5 Not offered this term This course is not o...,"Upon successful completion of this course, the..."
1,https://www.bcit.ca/courses/computers-and-the-...,Computers and the LawBLAW 3600,The course offers basic knowledge of Canadian ...,No prerequisites are required for this course.,4.0 Not offered this term This course is not o...,"Upon successful completion of the course, the ..."
2,https://www.bcit.ca/courses/dqm-with-python-ba...,DQM with PythonBABI 4005,​This course will introduce students to the co...,50% in BABI 3005,2.0 Not offered this term This course is not o...,"Upon successful completion of this course, the..."


In [7]:
df.to_csv("../data/courses.csv", index=False)
print("Saved to data/courses.csv")

Saved to data/courses.csv


In [8]:
df[["title", "prerequisites"]]

,title,prerequisites
0,Applied Data Management for AnalyticsBABI 4000,50% in BABI 3000
1,Computers and the LawBLAW 3600,No prerequisites are required for this course.
2,DQM with PythonBABI 4005,50% in BABI 3005


In [9]:
def clean_title(title):
    if title is None:
        return None

    # seperate course name from code
    for code in ["BABI", "BLAW"]:
        if code in title:
            parts = title.split(code)
            return parts[0].strip() + " (" + code + " " + parts[1].strip() + ")"
    
    return title

df["clean_title"] = df["title"].apply(clean_title)

df[["clean_title","prerequisites"]]

,clean_title,prerequisites
0,Applied Data Management for Analytics (BABI 4000),50% in BABI 3000
1,Computers and the Law (BLAW 3600),No prerequisites are required for this course.
2,DQM with Python (BABI 4005),50% in BABI 3005


In [10]:
prereq_map = df[["clean_title","prerequisites"]].copy()

prereq_map.columns = ["course","requires"]

prereq_map

,course,requires
0,Applied Data Management for Analytics (BABI 4000),50% in BABI 3000
1,Computers and the Law (BLAW 3600),No prerequisites are required for this course.
2,DQM with Python (BABI 4005),50% in BABI 3005


In [11]:
learning_df = df[["clean_title","learning_outcomes"]].copy()

learning_df.columns = ["course","learning_objectives"]

learning_df

,course,learning_objectives
0,Applied Data Management for Analytics (BABI 4000),"Upon successful completion of this course, the..."
1,Computers and the Law (BLAW 3600),"Upon successful completion of the course, the ..."
2,DQM with Python (BABI 4005),"Upon successful completion of this course, the..."


In [12]:
df.to_csv("../data/courses_cleaned.csv", index=False)
prereq_map.to_csv("../data/prerequisite_map.csv", index=False)
learning_df.to_csv("../data/learning_objectives.csv", index=False)

print("Saved cleaned datasets")

Saved cleaned datasets
